In [46]:
import pandas as pd
import numpy as np
import gc

In [47]:
print("Loading dataset...")

df = pd.read_csv(
    "final_model_dataset.csv",
    low_memory=False
)

print(f"Original Shape: {df.shape}")


Loading dataset...
Original Shape: (3145434, 110)


In [48]:
print("\nRemoving duplicate rows...")

before = len(df)

df = df.drop_duplicates()

after = len(df)

print(f"Removed {before - after} duplicates")
print(f"Shape: {df.shape}")



Removing duplicate rows...
Removed 0 duplicates
Shape: (3145434, 110)


In [49]:
address_cols = [
    'RegAddress.CareOf',
    'RegAddress.POBox',
    'RegAddress.AddressLine1',
    'RegAddress.AddressLine2',
    'RegAddress.PostTown',
    'RegAddress.County',
    'RegAddress.PostCode'
]

df.drop(
    columns=[c for c in address_cols if c in df.columns],
    inplace=True,
    errors='ignore'
)

print("Address columns removed")


Address columns removed


In [50]:
uri_cols = ['URI']

df.drop(
    columns=[c for c in uri_cols if c in df.columns],
    inplace=True,
    errors='ignore'
)

print("URI removed")

URI removed


In [51]:

previous_name_cols = [
    col for col in df.columns
    if "PreviousName_" in col
]

df.drop(
    columns=previous_name_cols,
    inplace=True,
    errors='ignore'
)

print(f"Removed {len(previous_name_cols)} PreviousName columns")

Removed 20 PreviousName columns


In [52]:
duplicate_features = [
    'IncorporationDate',
    'Accounts.NextDueDate',
    'Accounts.LastMadeUpDate',
    'ConfStmtNextDueDate'
]

df.drop(
    columns=[c for c in duplicate_features if c in df.columns],
    inplace=True,
    errors='ignore'
)

print("Duplicate engineered columns removed")

Duplicate engineered columns removed


In [53]:
missing_percent = (df.isnull().mean() * 100)

high_missing_cols = missing_percent[
    missing_percent > 80
].index.tolist()

print(f"\nColumns with >80% missing: {len(high_missing_cols)}")

df.drop(
    columns=high_missing_cols,
    inplace=True,
    errors='ignore'
)

print("High missing columns removed")


Columns with >80% missing: 32
High missing columns removed


In [54]:
date_columns = [
    'DissolutionDate',
    'Returns.NextDueDate',
    'Returns.LastMadeUpDate',
    'ConfStmtLastMadeUpDate',
    'incorporation_date',
    'accounts_next_due',
    'accounts_last_made_up',
    'conf_stmt_next_due'
]

for col in date_columns:

    if col in df.columns:

        df[col] = pd.to_datetime(
            df[col],
            errors='coerce'
        )

print("Date conversion completed")

/var/folders/9d/8dwzd1w92ynfdh1yvs1tz2s00000gn/T/ipykernel_58263/2992669534.py:16: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = pd.to_datetime(
/var/folders/9d/8dwzd1w92ynfdh1yvs1tz2s00000gn/T/ipykernel_58263/2992669534.py:16: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = pd.to_datetime(


Date conversion completed


In [55]:
numeric_cols = df.select_dtypes(
    include=[
        'int64',
        'float64',
        'int32',
        'float32'
    ]
).columns

for col in numeric_cols:

    median_val = df[col].median()

    df[col] = df[col].fillna(median_val)

print("Numeric missing values handled")

Numeric missing values handled


In [56]:
categorical_cols = df.select_dtypes(
    include=['object']
).columns

for col in categorical_cols:

    df[col] = df[col].fillna("Unknown")

print("Categorical missing values handled")

Categorical missing values handled


In [57]:
constant_cols = [
    col
    for col in df.columns
    if df[col].nunique(dropna=False) <= 1
]

df.drop(
    columns=constant_cols,
    inplace=True,
    errors='ignore'
)

print(f"Removed {len(constant_cols)} constant columns")

Removed 6 constant columns


In [58]:
for col in df.select_dtypes(include=['int64']).columns:
    df[col] = pd.to_numeric(
        df[col],
        downcast='integer'
    )

for col in df.select_dtypes(include=['float64']).columns:
    df[col] = pd.to_numeric(
        df[col],
        downcast='float'
    )

gc.collect()

print("Memory optimized")

Memory optimized


In [59]:
print("\nFINAL DATASET SUMMARY")
print("="*50)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

print("\nTotal Missing Values:")
print(df.isnull().sum().sum())

print("\nTop Missing Columns:")
print(
    (df.isnull().mean()*100)
    .sort_values(ascending=False)
    .head(20)
)


FINAL DATASET SUMMARY
Rows: 3,145,434
Columns: 40

Total Missing Values:
4149709

Top Missing Columns:
Returns.LastMadeUpDate       89.499955
accounts_last_made_up        24.712329
ConfStmtLastMadeUpDate       17.624976
accounts_next_due             0.070038
conf_stmt_next_due            0.019806
Returns.NextDueDate           0.000922
fast_growth_proxy             0.000000
num_mortgages_total           0.000000
accounts_overdue_days         0.000000
accounts_ever_late            0.000000
conf_stmt_overdue_days        0.000000
num_mortgages_outstanding     0.000000
has_changed_name              0.000000
has_active_mortgages          0.000000
is_full_accounts              0.000000
is_active                     0.000000
account_category              0.000000
is_dormant                    0.000000
is_micro                      0.000000
is_small                      0.000000
dtype: float64


In [61]:
print(df.shape)

print(df.isnull().sum().sum())

print(df.dtypes.value_counts())

(3145434, 40)
4149709
object            11
int8              11
datetime64[ns]     7
float32            5
int16              5
int32              1
Name: count, dtype: int64


In [62]:
missing = (df.isnull().mean()*100).sort_values(ascending=False)

print(missing[missing > 0])

Returns.LastMadeUpDate    89.499955
accounts_last_made_up     24.712329
ConfStmtLastMadeUpDate    17.624976
accounts_next_due          0.070038
conf_stmt_next_due         0.019806
Returns.NextDueDate        0.000922
dtype: float64


In [63]:
df.drop(
    columns=['Returns.LastMadeUpDate'],
    inplace=True,
    errors='ignore'
)

In [64]:
print(df.shape)

print(df.isnull().sum().sum())

print(df.dtypes.value_counts())

(3145434, 39)
1334547
object            11
int8              11
datetime64[ns]     6
float32            5
int16              5
int32              1
Name: count, dtype: int64


In [65]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3145434 entries, 0 to 3145433
Data columns (total 39 columns):
 #   Column                          Dtype         
---  ------                          -----         
 0   CompanyName                     object        
 1   CompanyNumber                   object        
 2   RegAddress.Country              object        
 3   CompanyCategory                 object        
 4   CompanyStatus                   object        
 5   CountryOfOrigin                 object        
 6   Accounts.AccountRefDay          float32       
 7   Accounts.AccountRefMonth        float32       
 8   Accounts.AccountCategory        object        
 9   Returns.NextDueDate             datetime64[ns]
 10  Mortgages.NumMortCharges        int16         
 11  Mortgages.NumMortOutstanding    int16         
 12  Mortgages.NumMortPartSatisfied  int8          
 13  Mortgages.NumMortSatisfied      int16         
 14  SICCode.SicText_1               object        
 15

In [66]:
(
    df['Accounts.AccountCategory'].fillna('Missing')
    ==
    df['account_category'].fillna('Missing')
).mean()

1.0

In [67]:
df.drop(
    columns=['Accounts.AccountCategory'],
    inplace=True
)

In [68]:
print(df.shape)

(3145434, 38)


In [69]:
output_file = "cleaned_companies_house_dataset.csv"

df.to_csv(
    output_file,
    index=False
)

print("\nCleaning Complete")
print(f"Saved as: {output_file}")
print(f"Final Shape: {df.shape}")


Cleaning Complete
Saved as: cleaned_companies_house_dataset.csv
Final Shape: (3145434, 38)
